# Analiza strezczeń literatury młodzieżowej

# import bibliotek

In [ ]:
import nltk
from nltk.corpus import PlaintextCorpusReader
import morfeusz2
import operator as op
import string 
import itertools as it
from nltk.tokenize import word_tokenize
import pandas as pd
import json
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

In [14]:
nltk.pathsec.ENFORCE = False

# utworzenie korpusu dokumentów

In [15]:
corpus_dir = "./Literature"
corpus = PlaintextCorpusReader(corpus_dir,r".*\txt")
file_names= corpus.fileids()

# Wstępne przygotowanie dokumentów

In [26]:
document_dict = {}
for  file_name in file_names:
    document_dict[file_name] = corpus.raw(file_name)
print(json.dumps(document_dict, indent = 4,ensure_ascii = False))

{}


In [29]:
lenghts = {}
for file_names in document_dict:
    lenghts[file_name]={
        "pre": len(word_tokenize(document_dict[file_name])),
    }
print(json.dumps(lenghts, indent=4 ,ensure_ascii = False))
    
    

{}


In [27]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding = "utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords = stopwords[4:]
stopwords_file.close()
stopwords


['vol',
 'o.o.',
 'mgr',
 'godz',
 'zł',
 'www',
 'pl',
 'ul',
 'tel',
 'hab',
 'prof',
 'inż',
 'dr',
 'i',
 'u',
 'aby',
 'albo',
 'ale',
 'ani',
 'aż',
 'bardzo',
 'bez',
 'bo',
 'bowiem',
 'by',
 'byli',
 'bym',
 'był',
 'była',
 'było',
 'były',
 'być',
 'będzie',
 'będą',
 'chce',
 'choć',
 'co',
 'coraz',
 'coś',
 'czy',
 'czyli',
 'często',
 'dla',
 'do',
 'gdy',
 'gdyby',
 'gdyż',
 'gdzie',
 'go',
 'ich',
 'im',
 'inne',
 'iż',
 'ja',
 'jak',
 'jakie',
 'jako',
 'je',
 'jednak',
 'jednym',
 'jedynie',
 'jego',
 'jej',
 'jest',
 'jeszcze',
 'jeśli',
 'jeżeli',
 'już',
 'ją',
 'kiedy',
 'kilku',
 'kto',
 'która',
 'które',
 'którego',
 'której',
 'który',
 'których',
 'którym',
 'którzy',
 'lat',
 'lecz',
 'lub',
 'ma',
 'mają',
 'mamy',
 'mi',
 'miał',
 'mimo',
 'mnie',
 'mogą',
 'może',
 'można',
 'mu',
 'musi',
 'na',
 'nad',
 'nam',
 'nas',
 'nawet',
 'nic',
 'nich',
 'nie',
 'niej',
 'nim',
 'niż',
 'no',
 'nowe',
 'np',
 'nr',
 'o',
 'od',
 'ok',
 'on',
 'one',
 'oraz',
 '

In [ ]:
def lematizite(text):
    morfor =morfeusz2.Morfeusz()
    segments = it.groupby(morfor.analyse(text), op.itemgetter(0,1))
    def disambleguate(group):
        pairs = ((len(descr),lema) for _,_, (_,_,lema,descr,_,_,) in group)
        perpl,lema = min(pairs)
        return lema.split(":")[0] if lema else None
    lemas = (disambleguate(group) for _,group in segments)
    return " ".join(filter(str.isalpha, lemas))

In [30]:
for file_name in document_dict:
    document_dict[file_name]= document_dict[file_name].lower()
    document_dict[file_name]= "".join([char for char in document_dict[file_name] if char not in string.punctuation])
    document_dict[file_name]= lematizite(document_dict[file_name])
    document_dict[file_name] ="".join([word for word in word_tokenize(document_dict[file_name], language="polish")if word not in stopwords])
    

In [33]:
for file_name in document_dict:
    lenghts[file_name]["post"] = len(word_tokenize(document_dict[file_name]))
print(json.dumps(lenghts, indent=4 ,ensure_ascii = False))
    
    

{}


In [ ]:
lenghts = pd.DataFrame.from_dict(lenghts,orient='index')
lenghts["change"]= lenghts["post"] - lenghts["pre"]
lenghts["change_percentage"] = ((lenghts["change"]/lenghts["pre"])*100).round(2)
lenghts

KeyError: 'post'

In [36]:
document_df = pd.DataFrame.from_dict(document_dict, orient="index", columns=["content"])
document_df

,content


In [ ]:
matrix_tf = CountVectorizer().fit_transform(document_df['content'])
matrix_tfidf =TfidfVectorizer().fit_transform(document_df['content'])


NameError: name 'CountVectorizer' is not defined

In [ ]:
#sparsity = (1 - matrix_tf.nnz)